# Generate The Pipeline Config

This notebook builds the tutorial config as ordinary Python dictionaries. Each editable code cell corresponds to one type of pipeline config: shared data knobs, loader managers, modeling components, HPO, train, evaluate, export, and inference.

The checked-in `pipeline/config.json` is what the run notebooks load. This notebook is the readable construction manual for that JSON.

> **Notebook memory:** Importing the scientific Python stack (for example PyTorch, NumPy, Matplotlib, and ZenML) can keep approximately 1 GiB of RAM assigned to this kernel for its lifetime. Python cannot safely unload native extension modules. **After finishing this tutorial, restart its kernel to clear imported libraries and release that RAM** (or shut down the kernel entirely). Restarting keeps the notebook open with a fresh, low-memory kernel. Do this before running several tutorial notebooks at once.

In [ ]:
from pathlib import Path
import json
import sys
from copy import deepcopy
from pprint import pprint

# Source-checkout bootstrap: this lets the notebook import PioneerML and the
# base plugin before either package has been installed in editable mode.
for root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    for src in (root / "src", root / "plugins" / "example_plugin" / "src"):
        if (src / "pioneerml_example_plugin").exists() or (src / "pioneerml").exists():
            src_text = str(src)
            if src_text not in sys.path:
                sys.path.insert(0, src_text)

from pioneerml_example_plugin.components_tutorial_examples.pipeline import load_config


## Shared Knobs

Edit this cell first when you want a larger run, a different data source, or different loader chunking. The remaining cells reuse these values.

In [ ]:
DATA_SOURCE = "sample_data/sensor_health.csv"
ROWS_PER_CHUNK = 64
BATCH_SIZE = 8
EXPORT_BATCH_SIZE = 1

TRAIN_FRACTION = 0.75
VAL_FRACTION = 0.25
SPLIT_SEED = 13

MAX_EPOCHS = 25
LIMIT_TRAIN_BATCHES = 20
LIMIT_VAL_BATCHES = 10

OUTPUT_ROOT = "tutorial_outputs"
MODEL_EXPORT_DIR = f"{OUTPUT_ROOT}/exports"
PREDICTION_DIR = f"{OUTPUT_ROOT}/predictions"


## Loader Manager Template

Edit this function when the input source, input backend, split settings, or default loader settings should change for every step. Individual steps add their own `loaders` entries below.

In [ ]:
def loader_manager_config(*, mode: str = "train", batch_size: int = BATCH_SIZE) -> dict:
    return {
        "type": "sensor_tutorial",
        "config": {
            "input_sources_spec": {
                "main_sources": [DATA_SOURCE],
                "optional_sources_by_name": {},
                "source_type": "file",
            },
            "input_backend": {
                "type": "sensor_csv",
                "config": {"rows_per_chunk": ROWS_PER_CHUNK},
            },
            "defaults": {
                "type": "sensor_health_loader",
                "config": {
                    "batch_size": int(batch_size),
                    "chunk_row_groups": 2,
                    "chunk_workers": 0,
                    "mode": mode,
                    "shuffle_batches": False,
                    "shuffle_within_batch": False,
                    "train_fraction": TRAIN_FRACTION,
                    "val_fraction": VAL_FRACTION,
                    "test_fraction": 0.0,
                    "sample_fraction": 1.0,
                    "split_seed": SPLIT_SEED,
                },
            },
            "loaders": {},
        },
    }


## Modeling Component Blocks

Edit this cell to swap registered component `type` names or tune their nested `config`. These blocks are reused by both HPO and the final train step.

In [ ]:
architecture = {
    "type": "sensor_health_mlp",
    "config": {
        "node_dim": 3,
        "edge_dim": 1,
        "hidden": 16,
        "dropout": 0.0,
        "output_dim": 1,
        "feature_mean": [68.0, 0.85, 3.30],
        "feature_std": [9.0, 0.65, 0.18],
    },
}

compiler = {"type": "sensor_health_noop", "config": {"tag": "tutorial"}}

module = {
    "type": "sensor_health_module",
    "config": {
        "loss": {
            "type": "sensor_health_bce",
            "config": {"positive_weight": 1.0},
        },
        "lr": 0.01,
        "weight_decay": 0.0,
        "max_step_history": 128,
        "max_epoch_history": 16,
    },
}

trainer = {
    "type": "sensor_health_trainer",
    "config": {
        "trainer_kwargs": {
            "max_epochs": MAX_EPOCHS,
            "limit_train_batches": LIMIT_TRAIN_BATCHES,
            "limit_val_batches": LIMIT_VAL_BATCHES,
        },
        "early_stopping": {
            "enabled": False,
            "type": "relative",
            "config": {
                "monitor": "train_loss",
                "mode": "min",
                "patience": 1,
                "min_delta": 0.0,
                "strict": False,
                "check_finite": True,
                "verbose": False,
            },
        },
    },
}

modeling_blocks = {
    "architecture": architecture,
    "compiler": compiler,
    "module": module,
    "trainer": trainer,
}


## HPO Step Config

Edit this cell to change the HPO plugin, trial count, objective, or search-space definition. It uses the same model and loader blocks as training so trial results are comparable.

In [ ]:
hpo_loader_manager = loader_manager_config(mode="train", batch_size=BATCH_SIZE)
hpo_loader_manager["config"]["loaders"] = {
    "train_loader": {"config": {"mode": "train", "split": "train", "shuffle_batches": True, "log_diagnostics": False}},
    "val_loader": {"config": {"mode": "train", "split": "val", "shuffle_batches": False, "log_diagnostics": False}},
}

hpo_step = {
    **deepcopy(modeling_blocks),
    "loader_manager": hpo_loader_manager,
    "hpo": {
        "type": "sensor_health_hpo",
        "config": {
            "enabled": True,
            "n_trials": 1,
            "direction": "minimize",
            "seed": 31,
            "study_name": "sensor_health_tutorial_bce",
            "storage": None,
            "fallback_dir": None,
            "allow_schema_fallback": True,
            "objective": {"type": "sensor_health_val_loss", "config": {}},
            "search_space": {
                "type": "sensor_health_search_space",
                "config": {
                    "search_space": {
                        "hidden": {"type": "categorical", "choices": [16, 32]},
                        "lr": {"type": "sensor_centered_float", "center": 0.01, "radius": 0.008, "log": True},
                    }
                },
            },
        },
    },
}


## Train Step Config

Edit this cell to change only final-training data behavior. For example, keep HPO fixed but train with a different batch size or shuffle policy.

In [ ]:
train_loader_manager = loader_manager_config(mode="train", batch_size=BATCH_SIZE)
train_loader_manager["config"]["loaders"] = {
    "train_loader": {"config": {"mode": "train", "split": "train", "shuffle_batches": True, "log_diagnostics": False}},
    "val_loader": {"config": {"mode": "train", "split": "val", "shuffle_batches": False, "log_diagnostics": False}},
}

train_step = {**deepcopy(modeling_blocks), "loader_manager": train_loader_manager}


## Evaluate Step Config

Edit this cell to change evaluator plugin, metrics, plots, threshold, or the loader used for evaluation.

In [ ]:
evaluate_loader_manager = loader_manager_config(mode="train", batch_size=BATCH_SIZE)
evaluate_loader_manager["config"]["loaders"] = {
    "test_loader": {"config": {"mode": "train", "split": "val", "shuffle_batches": False, "log_diagnostics": False}},
}

evaluate_step = {
    "evaluator": {
        "type": "sensor_health_evaluator",
        "config": {
            "threshold": 0.5,
            "metrics": ["sensor_health_accuracy"],
            "plots": ["sensor_health_score_histogram"],
            "plot_path": f"{OUTPUT_ROOT}/sensor_score_histogram.png",
        },
    },
    "loader_manager": evaluate_loader_manager,
}


## Export Step Config

Edit this cell to change exporter plugin, output directory, filename prefix, or export-time loader behavior.

In [ ]:
export_loader_manager = loader_manager_config(mode="train", batch_size=EXPORT_BATCH_SIZE)
export_loader_manager["config"]["defaults"]["config"]["chunk_row_groups"] = 1
export_loader_manager["config"]["loaders"] = {
    "export_loader": {"config": {"mode": "train", "shuffle_batches": False, "log_diagnostics": False}},
}

export_step = {
    "exporter": {
        "type": "sensor_health_state_dict",
        "config": {
            "enabled": True,
            "export_dir": MODEL_EXPORT_DIR,
            "filename_prefix": "sensor_health",
            "prefer_cuda": False,
        },
    },
    "loader_manager": export_loader_manager,
}


## Inference Config

Edit this cell to change the model-handle plugin, batch executor, inference loader, writer, output backend, or prediction path. The run notebook patches `model_path` after training exports a fresh local artifact.

In [ ]:
inference_loader_manager = loader_manager_config(mode="inference", batch_size=BATCH_SIZE)
inference_loader_manager["config"]["loaders"] = {
    "inference_loader": {"config": {"mode": "inference", "shuffle_batches": False, "log_diagnostics": False}},
}

model_handle_builder = {
    "model_handle": {
        "type": "sensor_health_state_dict",
        "config": {"model_path": f"{MODEL_EXPORT_DIR}/model.pt"},
    }
}

inference_step = {
    "runtime": {"prefer_cuda": False},
    "batch_executor": {
        "type": "sensor_health_batch_executor",
        "config": {"require_finite": True},
    },
    "writer": {
        "type": "sensor_health_writer",
        "config": {
            "output_backend": {"type": "sensor_jsonl", "config": {}},
            "fallback_output_dir": PREDICTION_DIR,
            "output_dir": PREDICTION_DIR,
            "output_path": f"{PREDICTION_DIR}/sensor_predictions.jsonl",
            "streaming": False,
            "write_timestamped": False,
            "timestamp": None,
            "writer_params": {},
        },
    },
    "loader_manager": inference_loader_manager,
}

inference_config = {
    "model_handle_builder": model_handle_builder,
    "inference": inference_step,
}


## Assemble The Full Config

This is the shape consumed by the standard PioneerML pipelines: `training_pipeline` reads `full_config["training"]`, and `inference_pipeline` reads `full_config["inference"]`.

In [ ]:
training_config = {
    "hpo": hpo_step,
    "train": train_step,
    "evaluate": evaluate_step,
    "export": export_step,
}

full_config = {"training": training_config, "inference": inference_config}

print("Top-level sections:", list(full_config))
print("Training steps:", list(training_config))
print("Inference steps:", list(inference_config))


## Compare Against The Checked-In JSON

The comparison should be `True`. If you edit any cell above, this becomes a quick way to see that your generated config intentionally diverges from `pipeline/config.json`.

In [ ]:
checked_in_config = load_config()
print("Generated config matches checked-in config:", full_config == checked_in_config)

# Uncomment to inspect one section without dumping the entire config.
# pprint(full_config["training"]["train"])


## Optional Save

Leave this commented unless you intentionally want to replace the checked-in tutorial config after editing the cells above.

In [ ]:
# config_path = Path("../pipeline/config.json").resolve()
# config_path.write_text(json.dumps(full_config, indent=2) + "\n", encoding="utf-8")
# print(config_path)
